In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import netCDF4

In [2]:
f = netCDF4.Dataset('data/cpm-20250603.nc')
print(f)

<class 'netCDF4.Dataset'>
root group (NETCDF4 data model, file format HDF5):
    dimensions(sizes): LI(1336), TI(130), GE(20222)
    variables(dimensions): <class 'str'> LI(LI), float64 TI(TI), <class 'str'> GE(GE), float32 layer(TI, LI, GE)
    groups: 


In [3]:
exp = f.variables['layer']
print(exp)

<class 'netCDF4.Variable'>
float32 layer(TI, LI, GE)
    name: layer
    _FillValue: 1e+32
unlimited dimensions: 
current shape = (130, 1336, 20222)
filling on


In [ ]:
n_lineages = exp.shape[1]
n_genes = exp.shape[2]

In [5]:
# Initialize empty DataFrame with proper dimensions
lineage_names = f.variables['LI'][:]
gene_names = f.variables['GE'][:]
exp_mat = np.zeros((n_lineages, n_genes))

# Process data in chunks along the time dimension
chunk_size = 10  # Process 10 time points at a time
n_time_points = exp.shape[0]

for start_idx in tqdm(range(0, n_time_points, chunk_size)):
    end_idx = min(start_idx + chunk_size, n_time_points)
    
    # Load chunk of data
    chunk_data = exp[start_idx:end_idx, :, :]
    chunk_data = np.nan_to_num(chunk_data, nan=0.0)
    
    # Add to running sum
    exp_mat += np.sum(chunk_data, axis=0)

# Calculate mean by dividing by total time points
exp_mat = exp_mat / n_time_points

# Create DataFrame
exp_df = pd.DataFrame(exp_mat, columns=gene_names, index=lineage_names)
exp_df.index.name = 'lineage'


100%|██████████| 13/13 [04:47<00:00, 22.12s/it]


In [7]:
exp_df.head()

,2L52.1,2RSSE.1,4R79.2,6R55.2,AC3.12,AC3.5,AC7.3,AC8.10,AC8.11,AC8.12,...,ztf-7,ztf-8,ztf-9,zwl-1,zyg-1,zyg-11,zyg-12,zyg-8,zyg-9,zyx-1
lineage,,,,,,,,,,,,,,,,,,,,,
ABa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ABal,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ABala,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ABalaa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ABalaaa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
# for each lineage, calculate how many genes are expressed in exp_df
def count_expressed_genes(lineage_series):
    return (lineage_series > 0).sum()
# Apply the function to each lineage
res = exp_df.apply(count_expressed_genes, axis=1)
res.sort_values(ascending=False, inplace=True)
res.name = 'n_expressed_genes'
# save the result to a CSV file
res.to_csv('data/n_expressed_genes.csv')

In [20]:
# get expression of mlc-3 gene and ABarppppaa lineage
ABarppppaa_lineage_exp = exp_df.loc["ABarppppaa", "mlc-3"]

In [21]:
ABarppppaa_lineage_exp

np.float64(192.68708143967848)